## Fetching dataset

In [42]:
!pip install ucimlrepo

In [43]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

# data (as pandas dataframes)
X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

# metadata
breast_cancer_wisconsin_diagnostic.metadata

{'uci_id': 17,
 'name': 'Breast Cancer Wisconsin (Diagnostic)',
 'repository_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic',
 'data_url': 'https://archive.ics.uci.edu/static/public/17/data.csv',
 'abstract': 'Diagnostic Wisconsin Breast Cancer Database.',
 'area': 'Health and Medicine',
 'tasks': ['Classification'],
 'characteristics': ['Multivariate'],
 'num_instances': 569,
 'num_features': 30,
 'feature_types': ['Real'],
 'demographics': [],
 'target_col': ['Diagnosis'],
 'index_col': ['ID'],
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 1993,
 'last_updated': 'Fri Nov 03 2023',
 'dataset_doi': '10.24432/C5DW2B',
 'creators': ['William Wolberg',
  'Olvi Mangasarian',
  'Nick Street',
  'W. Street'],
 'intro_paper': {'ID': 230,
  'type': 'NATIVE',
  'title': 'Nuclear feature extraction for breast tumor diagnosis',
  'authors': 'W. Street, W. Wolberg, O. Mangasarian',
  'venue': 'Electronic imaging',
  'yea

In [44]:
print(X.head())

   radius1  texture1  perimeter1   area1  smoothness1  compactness1  \
0    17.99     10.38      122.80  1001.0      0.11840       0.27760   
1    20.57     17.77      132.90  1326.0      0.08474       0.07864   
2    19.69     21.25      130.00  1203.0      0.10960       0.15990   
3    11.42     20.38       77.58   386.1      0.14250       0.28390   
4    20.29     14.34      135.10  1297.0      0.10030       0.13280   

   concavity1  concave_points1  symmetry1  fractal_dimension1  ...  radius3  \
0      0.3001          0.14710     0.2419             0.07871  ...    25.38   
1      0.0869          0.07017     0.1812             0.05667  ...    24.99   
2      0.1974          0.12790     0.2069             0.05999  ...    23.57   
3      0.2414          0.10520     0.2597             0.09744  ...    14.91   
4      0.1980          0.10430     0.1809             0.05883  ...    22.54   

   texture3  perimeter3   area3  smoothness3  compactness3  concavity3  \
0     17.33      184.60 

In [45]:
y.head()

,Diagnosis
0,M
1,M
2,M
3,M
4,M


In [46]:
X.shape
X.info()

print("-----------------------------------------")

y.shape
y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavit

##Preprocessing

In [70]:
import pandas as pd
import numpy as np
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.linear_model
import sklearn.metrics
import sklearn.tree

In [48]:
#encode the M to 1 and B to 0
y_encoded = y['Diagnosis'].map({'M':1, 'B':0})
y_encoded.head(5)

,Diagnosis
0,1
1,1
2,1
3,1
4,1


In [49]:
print(y_encoded.value_counts())
print(y_encoded.unique())

Diagnosis
0    357
1    212
Name: count, dtype: int64
[1 0]


In [50]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
  )

Stratification ensures that both training and test sets preserve the original class distribution, which is especially important in medical datasets where recall and precision for the minority (malignant) class are critical.

In [51]:
print("--------Shape/rows--------------")
print(X_train.shape[0])
print(X_test.shape[0])
print(y_train.shape[0])
print(y_test.shape[0])
print("------Count-------")
print(y_train.value_counts())
print(y_test.value_counts())

--------Shape/rows--------------
455
114
455
114
------Count-------
Diagnosis
0    285
1    170
Name: count, dtype: int64
Diagnosis
0    72
1    42
Name: count, dtype: int64


## FEATURE SCALING

In [54]:
sclaer = sklearn.preprocessing.StandardScaler()
X_train_scaled = sclaer.fit_transform(X_train)
X_test_scaled = sclaer.transform(X_test)

The scaler must be fit only on the training data because it learns statistics such as mean and standard deviation.
Fitting the scaler on the test data would leak information from the test set into the training process, violating the assumption that test data is unseen and leading to an overly optimistic estimate of generalization performance.

## Logistic Regression

In [58]:
model = sklearn.linear_model.LogisticRegression(max_iter=100, random_state=42)
model.fit(X_train_scaled, y_train)
model.score(X_train_scaled, y_train)

0.9868131868131869

In [60]:
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

print("Training predictions:", y_train_pred[:10])
print("Test predictions:", y_test_pred[:10])

Training predictions: [1 0 0 1 1 1 0 1 1 1]
Test predictions: [0 1 0 1 1 0 1 0 0 0]


In [66]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print("------------Accuracy-----------")
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Logistic Regression :: Training Accuracy:", train_accuracy)
print("Logistic Regression :: Test Accuracy:", test_accuracy)
print("------------Precision-----------")
train_precision = precision_score(y_train, y_train_pred)
test_precision = precision_score(y_test, y_test_pred)
print("Logistic Regression :: Training Precision:", train_precision)
print("Logistic Regression :: Test Precision:", test_precision)
print("------------Recall-----------")
train_recall = recall_score(y_train, y_train_pred)
test_recall = recall_score(y_test, y_test_pred)
print("Logistic Regression :: Training Recall:", train_recall)
print("Logistic Regression :: Test Recall:", test_recall)

------------Accuracy-----------
Logistic Regression :: Training Accuracy: 0.9868131868131869
Logistic Regression :: Test Accuracy: 0.9649122807017544
------------Precision-----------
Logistic Regression :: Training Precision: 1.0
Logistic Regression :: Test Precision: 0.975
------------Recall-----------
Logistic Regression :: Training Recall: 0.9647058823529412
Logistic Regression :: Test Recall: 0.9285714285714286


Logistic Regression shows good generalization with a small train–test gap, indicating low variance and stable performance on unseen data.

In [67]:
print("------------F1 Score-----------")
train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)
print("Logistic Regression :: Training F1 Score:", train_f1)
print("Logistic Regression :: Test F1 Score:", test_f1)

print("------------ConfusionMatrix-----------")
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
print("Logistic Regression :: Training Confusion Matrix:")
print(train_confusion_matrix)
print("Logistic Regression :: Test Confusion Matrix:")
print(test_confusion_matrix)

------------F1 Score-----------
Logistic Regression :: Training F1 Score: 0.9820359281437125
Logistic Regression :: Test F1 Score: 0.9512195121951219
------------ConfusionMatrix-----------
Logistic Regression :: Training Confusion Matrix:
[[285   0]
 [  6 164]]
Logistic Regression :: Test Confusion Matrix:
[[71  1]
 [ 3 39]]


The model achieves higher precision than recall for the malignant class, indicating that it makes very few false positive predictions but still produces some false negatives, which is more critical in a cancer diagnosis task.

Conclusion for Logistic Regression:

* Small train–test gap → good generalization

* High precision → reliable positive predictions

* Slightly lower recall → room for improvement in catching all malignant cases

* Overall → well-balanced baseline model

## Decision Tree

In [79]:
model = sklearn.tree.DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
print("Modal Score: ", model.score(X_train, y_train))

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

Modal Score:  1.0


In [81]:
print("--------Accuracy----------")
DT_Train_accuracy = accuracy_score(y_train, y_train_pred)
print("Decision Tree :: Training Accuracy:", DT_Train_accuracy)
DT_Test_accuracy = accuracy_score(y_test, y_test_pred)
print("Decision Tree :: Test Accuracy:", DT_Test_accuracy)

print("--------Precision----------")
DT_Train_precision = precision_score(y_train, y_train_pred)
print("Decision Tree :: Training Precision:", DT_Train_precision)
DT_Test_precision = precision_score(y_test, y_test_pred)
print("Decision Tree :: Test Precision:", DT_Test_precision)

print("--------Recall----------")
DT_Train_recall = recall_score(y_train, y_train_pred)
print("Decision Tree :: Training Recall:", DT_Train_recall)
DT_Test_recall = recall_score(y_test, y_test_pred)
print("Decision Tree :: Test Recall:", DT_Test_recall)

print("--------F1 Score----------")
DT_Train_f1 = f1_score(y_train, y_train_pred)
print("Decision Tree :: Training F1 Score:", DT_Train_f1)
DT_Test_f1 = f1_score(y_test, y_test_pred)
print("Decision Tree :: Test F1 Score:", DT_Test_f1)

print("--------Confusion Matrix----------")
DT_Train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
print("Decision Tree :: Training Confusion Matrix:")
print(DT_Train_confusion_matrix)
DT_Test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
print("Decision Tree :: Test Confusion Matrix:")
print(DT_Test_confusion_matrix)

--------Accuracy----------
Decision Tree :: Training Accuracy: 1.0
Decision Tree :: Test Accuracy: 0.9298245614035088
--------Precision----------
Decision Tree :: Training Precision: 1.0
Decision Tree :: Test Precision: 0.9047619047619048
--------Recall----------
Decision Tree :: Training Recall: 1.0
Decision Tree :: Test Recall: 0.9047619047619048
--------F1 Score----------
Decision Tree :: Training F1 Score: 1.0
Decision Tree :: Test F1 Score: 0.9047619047619048
--------Confusion Matrix----------
Decision Tree :: Training Confusion Matrix:
[[285   0]
 [  0 170]]
Decision Tree :: Test Confusion Matrix:
[[68  4]
 [ 4 38]]


* Overfitting = high variance = large train–test gap
* Underfitting = high bias = both accuracies low
* Good fit = small gap + both reasonably high

Training accuracy: 1.0 (100%)

Test accuracy: 0.9298 (~93%)

Train–test gap: ~7% → much larger than Logistic Regression


> Thus it is OVERFITTING




# Conclusion

In this experiment, Logistic Regression and Decision Tree classifiers were trained to predict breast cancer diagnosis. Logistic Regression showed a small difference between training accuracy (98.7%) and test accuracy (96.5%), indicating good generalization and low variance. Its high precision (0.975) and reasonably high recall (0.93) for malignant tumors suggest that it makes reliable predictions while missing relatively few critical cases. This indicates that the model is slightly biased but generalizes well to unseen data, making it a strong baseline for this task.

In contrast, the Decision Tree achieved perfect training performance (100% accuracy) but showed a noticeable drop in test accuracy (92.98%), along with lower precision and recall for malignant cases. This large generalization gap indicates overfitting due to high variance, as the model memorized the training data but failed to generalize effectively. Additionally, the Decision Tree produced more false negatives on the test set, which is undesirable in a medical diagnosis context. Overall, Logistic Regression provided a better bias–variance tradeoff and more reliable generalization than the Decision Tree for this dataset.